# TS-SatFire: Kriging 결측 복원 → 로지스틱 GAM 다음 날 확산 예측

사건: **US_2021_NM3323810847220210520**, New Mexico, FP 원래 시험 사건, `label_sel=0`.

**목적:** 실제 사건 자료를 먼저 감사한 뒤, 두 입력 날짜 t−1,t로 다음 날 새로 기록된 AF/BA 누적 영역을 예측합니다. 이 사건의 앞부분을 학습에 사용하므로 **동일 사건 내 시간 전이 탐색 실험**입니다. 공식 FP 시험 성능·독립 사건 일반화·논문 SwinUNETR 재현이 아닙니다. 처음부터 전 기간을 보고 모델을 조정하면 이후 점수도 탐색 결과로 취급합니다.

기본값은 실제 HPC 경로입니다. `DEMO_MODE=True`는 별도 합성 GeoTIFF를 생성하고 전체 경로를 검증합니다. 실데이터가 없다고 자동으로 demo로 바꾸지 않습니다. 원자료는 읽기만 합니다.

**타겟:** 확인 가능한 `C(t+1) & ~C(t)`, C는 사건 시작부터 누적 AF∪BA. 순수 BA 증가, 실제 발화 확률, 건강 risk map이 아닙니다. unknown 라벨은 보간하지 않습니다.


## 0. 환경

노트북과 `satfire_pipeline.py`, `satfire_demo.py`를 같은 폴더에 복사합니다. CPU 작업이며 GPU는 필요 없습니다. 로그인 노드 대신 HPC interactive compute allocation에서 실행하십시오. 패키지는 별도 환경에서 설치합니다(아래 명령은 터미널 예시).

```bash
conda create -n satfire-gam python=3.11 -y
conda activate satfire-gam
python -m pip install -r requirements-satfire.txt
python -m ipykernel install --user --name satfire-gam --display-name "TS-SatFire GAM"
```

처음에는 기본 feature subset으로 시작하고, `inventory.csv`와 시간 가용성 확인 후 확장합니다. runtime package versions와 설정을 결과 폴더에 저장합니다.

In [ ]:
from pathlib import Path
import os, sys, json, importlib.metadata, hashlib
from datetime import datetime
import numpy as np
import pandas as pd
from IPython.display import display, Image
# Notebook working directory should be the bundle folder, or set BUNDLE_DIR.
BUNDLE_DIR = Path(os.environ.get("SATFIRE_NOTEBOOK_BUNDLE", Path.cwd()))
assert (BUNDLE_DIR / "satfire_pipeline.py").exists(), "Set BUNDLE_DIR to the notebook/helper folder"
sys.path.insert(0, str(BUNDLE_DIR))
import satfire_pipeline as sp

DEMO_MODE = False
CFG = {
    "raw_root": os.environ.get("TS_SATFIRE_RAW_ROOT", "/home/jlc3q/data/SatFire/ts-satfire"),
    "event_id": "US_2021_NM3323810847220210520",
    "seed": 20261119,
    "crop": None,  # Full raster. Optional (row_offset, col_offset, height, width).
    # This is NOT the legacy fixed 256x256 crop.
    "features": ["ndvi", "evi2", "i4", "tmax", "specific_humidity", "erc",
                 "wind_speed", "wind_direction", "slope", "elevation"],
    "value_rules": {},  # Example ONLY after metadata verification: {"ndvi":{"scale":0.0001,"range":[-1,1]}}
    "label_nan_policy": "strict_unknown",  # Alternative: legacy_nan_zero, after verifying sparse encoding.
    "label_policy_reason": "Unknown remains unknown; finite raw values >0 are recorded fire.",
    "train_pixel_fraction": 0.05,  # Uniform across all train dates/classes, NOT 50:50 class balancing.
    "kriging": {"max_support": 400, "min_support": 8, "neighbors": 24, "max_distance_km": 15.0},
    "lambdas": [1.0, 10.0, 100.0], "n_splines": 6, "max_iter": 250,
}
# Complete these gates only AFTER reviewing the audit and source timing.
SCHEMA_AND_LABELS_REVIEWED = False
PREDICTOR_TIMING_REVIEWED = False
TIMING_NOTE = ""  # NDVI compositing/available-at, gridMET daily timing, GFS issuance etc.
# Even when reviewed, this starts as retrospective recorded-label analysis, not operational deployment.
RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT = BUNDLE_DIR / "runs" / (("synthetic_" if DEMO_MODE else "real_") + RUN_TAG)
OUT.mkdir(parents=True, exist_ok=False)
if DEMO_MODE:
    from satfire_demo import make_demo
    CFG["raw_root"] = str(OUT / "synthetic_raw")
    CFG["event_id"] = "SYNTHETIC_NOT_NM332"
    make_demo(Path(CFG["raw_root"]), CFG["event_id"])
    CFG.update(features=["ndvi", "i4", "tmax"], train_pixel_fraction=0.5,
               lambdas=[1.0,10.0], n_splines=5)
    CFG["kriging"].update(max_support=120, neighbors=16, max_distance_km=8)
    SCHEMA_AND_LABELS_REVIEWED = PREDICTOR_TIMING_REVIEWED = True
    TIMING_NOTE = "Synthetic fixture; no real-world forecast validation."
versions={p:importlib.metadata.version(p) for p in ["numpy","scipy","pandas","rasterio","scikit-learn","pykrige","pygam","matplotlib"]}
sp.dump(OUT/"provenance.json", {"config":CFG,"demo":DEMO_MODE,"versions":versions,
    "helper_sha256":hashlib.sha256((BUNDLE_DIR/"satfire_pipeline.py").read_bytes()).hexdigest(),
    "scope":"within-event chronological exploratory; original test event reused for development"})
print(OUT)


## 1. 실제 파일 감사 — 이 표를 확인하기 전 학습하지 않습니다

확인한 기존 코드 계약: `VIIRS_Day` 밴드 1–6=I1/I2/I3/I4/I5/M11, 7=AF, 8=BA. `FirePred`는 19밴드이며 1=NDVI_last, 2=EVI2_last. 실제 사건의 descriptions·scale/offset·nodata가 맞는지 아래 표로 확인합니다. 원본 27채널 stack의 밴드 번호와 혼용하지 않습니다.

NaN을 cloud라고 단정하지 않습니다. 별도 QA가 없다면 여기서는 **GeoTIFF nodata/NaN 및 사용자가 명시한 invalid 값의 복원**만 수행합니다. 이미 0으로 채운 NPY가 아닌 raw GeoTIFF를 사용합니다. 구름 QA가 확인되면 원자료 복사본/읽기 단계에서 마스크를 적용하고 근거를 기록해야 합니다.

**중요:** 원본 legacy 코드는 라벨 NaN을 0으로 치환합니다. 기본 strict 정책에서는 NaN이 많으면 학습 가능한 타겟이 없어 중단할 수 있습니다. sparse label의 NaN이 배경임을 데이터 생산 규칙으로 확인한 경우에만 `legacy_nan_zero`와 이유를 선택합니다. 파일 누락·격자 불일치는 어느 정책에서도 unknown입니다.

In [ ]:
files, days, meta, inventory = sp.audit(CFG, OUT)
display(inventory.head(12))
display(inventory.groupby(["product","band","description"], dropna=False)["valid_fraction"].agg(["min","mean","max"]))
print((OUT/"audit.json").read_text())
print("FirePred one-based band mapping:", {n:v for n,v in sp.SPECS.items() if v[0]=="FirePred"})


## 2. 실행 승인 조건과 chronological split

`SCHEMA_AND_LABELS_REVIEWED`와 `PREDICTOR_TIMING_REVIEWED`를 확인 후 True로 설정하고 `TIMING_NOTE`를 작성합니다. NDVI/EVI 합성 기간의 종료/이용 가능일이 t 이후인지 확인합니다. “t의 관측 완료 후 다음 날”을 기준으로 기상 일평균의 가용성도 점검합니다. QA·라벨 검토 없이 강제로 실행한 결과는 사용하지 않습니다.

전체 연속 3일 창을 시간 순서대로 train 약55%, validation 다음23%, test 마지막22%로 나누고 경계마다 2개 창을 제거하여 입력/타겟 날짜 중첩을 막습니다. 과거 화재의 누적 상태는 이후 시점에서 합법적인 과거 정보입니다. 같은 사건의 공간 상관은 남으므로 독립 사건 시험이 아닙니다. `ANCHOR_DATE` 하나는 시각화에는 충분하지만 GAM 학습에는 여러 날짜의 양성/음성 사례가 필요합니다.

In [ ]:
assert SCHEMA_AND_LABELS_REVIEWED, "Review inventory and label encoding, then set the schema gate."
assert PREDICTOR_TIMING_REVIEWED and TIMING_NOTE.strip(), "Review predictor availability and document TIMING_NOTE."
assert CFG["label_policy_reason"].strip()
assert 0 < CFG["train_pixel_fraction"] <= 1
issues=json.loads((OUT/"audit.json").read_text())["issues"]
assert not issues, f"Fix file/grid problems before fitting: {issues[:5]}"
assert not json.loads((OUT/"audit.json").read_text())["missing_FirePred"], "Missing FirePred dates: repair inputs or explicitly redesign windows; no zero filling."
sp.dump(OUT/"review_decisions.json", {"timing":TIMING_NOTE,"label_policy":CFG["label_nan_policy"],
    "label_reason":CFG["label_policy_reason"],"schema_reviewed":True})
groups=sp.split_windows(days)
display(pd.DataFrame([{"split":k,"windows":len(v),"first_input":v[0][0],"last_target":v[-1][2]} for k,v in groups.items()]))


## 3. 지표별 복원·블록 마스킹 검증·특징 구성

각 날짜는 그 날짜의 유효 픽셀만 사용합니다. 좌표는 local metric CRS(km)로 변환하고 spherical variogram을 적합합니다. 계산량 제한으로 최대 400개 유효점과 moving neighborhood를 사용하므로 전 자료 exact kriging이 아닙니다. 먼 결측·상수장·지원 부족·계산 실패는 NaN으로 남깁니다. kriging variance는 조건부 지표 복원 분산이며 최종 화재 확률의 신뢰구간이 아닙니다.

풍향·사면 방향은 sin/cos로 변환 후 각각 처리합니다. landcover 정수 코드는 kriging/GAM spline에 넣지 않도록 차단합니다. 모든 연속형 지표 후보를 목록에서 선택할 수 있지만 I4 등 열적 hotspot을 평활화하는 영향은 별도 평가해야 합니다. 원시 I4를 사용하며 legacy의 누적 최대 열영상 전처리를 재현하지 않습니다.

각 split의 첫 t에서 소/중/대 블록을 숨겨 kriging과 IDW를 같은 위치에서 비교합니다. 이는 최소 diagnostic이며 실구름 아래의 정답 검증은 아닙니다. 정상 NDVI와 복원 NDVI를 각각 보존합니다.

각 샘플에는 t−1,t 지표와 현재 화재까지 거리, 주변 화재 비율, 최근 증가 비율을 넣습니다. ROI 전체 중 라벨이 알려진 아직 미기록 픽셀을 평가하며, 정답을 보고 fire-front buffer를 고르지 않습니다. 거리는 grid pixels로 저장하며 격자가 고정된 이 사건 내 실험용입니다.

In [ ]:
tables, feature_names, states = sp.build_tables(files, days, meta, CFG, OUT)
display(pd.read_csv(OUT/"restoration_validation.csv"))
display(pd.read_csv(OUT/"split_manifest.csv"))
display(pd.DataFrame({k:{"rows":len(v),"positives":int(v.y.sum()),"prevalence":v.y.mean()} for k,v in tables.items()}).T)


## 4. 비교 모델 학습 — validation만 사용해 선택

1. `median_gam`: 원시 결측을 training median + missing indicator로 처리한 GAM.
2. `kriging_logistic`: kriging 후 동일 fallback을 쓰는 선형 logit 비교군.
3. `kriging_gam`: kriging 후 비선형 additive spline logistic GAM.
4. training prevalence 상수 확률 기준.

kriging으로 복원되지 않은 값은 **훈련 median fallback**임을 명시합니다. 전부 kriging으로 복원했다고 보고하지 않습니다. 기본 전체 사례율을 유지하는 uniform sampling이며 class_weight나 50:50 undersampling을 쓰지 않습니다. λ는 validation log loss, 지도 임계값은 validation F1으로 선택합니다. scaler·median·feature selection은 train에서만 학습합니다. calibration 재보정은 여기서 수행하지 않습니다.

지표: PR-AUC(average precision), ROC-AUC, Brier, log loss, thresholded precision/recall/F1/IoU. pooled와 날짜별 및 원관측/결측입력 그룹을 함께 출력합니다. 이 지표는 미기록 eligible 픽셀의 pointwise 점수이며 논문의 전체 crop macro 점수와 동일하지 않습니다.

In [ ]:
predictions, metrics, models = sp.fit_evaluate(tables, feature_names, CFG, OUT)
display(metrics[metrics.split=="test_pooled"])
display(metrics[metrics.split.isin(["observed_inputs","missing_inputs"])])


## 5. 결과 지도 및 calibration

GeoTIFF: 발생 확률 / 관측 신규 기록 타겟 / 오분류 / 입력 결측 비율. 오분류 값 0=TN,1=FP,2=FN,3=TP. 이미 기록된 화재와 unknown은 확률 평가 지도에서 NaN이며 0%와 다릅니다. 이 지도는 관측 타겟을 평가할 수 있는 영역의 지도입니다. 완전한 operational forecast 배포 지도는 별도 지원 영역 정의가 필요합니다.

PNG와 CSV를 확인하고 구름 복원 오차·확률 오차를 별도로 보고합니다. 확률 map을 health risk 또는 물리적 전파 확률로 확대 해석하지 않습니다. 지도 모양만으로 성공을 판정하지 않습니다.

In [ ]:
sp.plot_results(predictions, metrics, states, meta, models, OUT)
display(Image(filename=str(OUT/"calibration.png")))
PANELS=sorted(OUT.glob("*_panels.png"))
if PANELS: display(Image(filename=str(PANELS[0])))
sp.dump(OUT/"completion.json", {"completed":True,"synthetic":DEMO_MODE,
    "scope":"single-event temporal exploratory test", "n_test_pixels":len(predictions),
    "hpc_real_validation":not DEMO_MODE,"remaining_limits":["label observation delay", "within-event dependence", "no final probability uncertainty propagation", "no independent-event validation", "no automatic cloud QA"]})
print("Artifacts:", OUT)


## 해석 체크리스트

- 복원된 비율과 미복원 비율, 지표별 MAE/RMSE/bias·조건부 구간 coverage를 함께 읽습니다.
- kriging GAM이 median GAM보다 Brier/log loss/calibration을 개선했는지 확인합니다. 개선이 없으면 kriging을 필수라고 주장하지 않습니다.
- test에서 모델/threshold를 수정하면 그 test는 development로 전환합니다.
- 실제 label NaN의 의미·자료 합성/발행 시각이 미확인이라면 결과는 해석하지 않습니다.
- 다음 단계는 다른 train/validation 사건에서 학습하고 이 사건을 제외한 독립 사건 시험 설계입니다. 이 노트북 자체로 agent 간 협업 성능이 검증되지는 않습니다.

### 근거

- TS-SatFire: https://arxiv.org/html/2412.11555v1 / https://doi.org/10.1038/s41597-025-06271-3
- PyKrige: https://geostat-framework.readthedocs.io/projects/pykrige/en/stable/generated/pykrige.ok.OrdinaryKriging.html
- pyGAM: https://pygam.readthedocs.io/en/stable/reference/_autosummary/pygam.pygam.LogisticGAM.html
- 기존 코드 감사와 HPC 실행 안내: 동봉 `README_HPC.md`, `SOURCE_AUDIT.md`.
